In [ ]:
# Analisis de indicadores de poblacion - Ano 2020
# Practica del modulo 24: visualizacion de datos con Python

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 1. Leer y revisar los datos
# Carpeta donde se encuentra el archivo
carpeta = Path.cwd() / "data-analysis" / "datasets" / "population-2020"
carpeta.mkdir(parents=True, exist_ok=True)

# Ruta del archivo CSV
archivo = carpeta / "SYB66_246_202310_Population Growth, Fertility and Mortality Indicators.csv"

# Cargar los datos
if not archivo.exists():
    try:
        import gdown
        gdown.download(id="1TN-qH-2YY5HqMp4iGF77H6JPLZYiB5TT", output=str(archivo), quiet=False)
    except ImportError:
        raise FileNotFoundError(f"Download the public dataset to {archivo}, or install gdown") from None

df = pd.read_csv(archivo, encoding="latin1", skiprows=1)

# Renombrar la columna de paises
df = df.rename(columns={'Unnamed: 1': 'Pais'})

# Convertir Value a numero para poder calcular promedios y correlaciones.
df['Value'] = pd.to_numeric(
    df['Value'].astype(str).str.replace(',', '', regex=False),
    errors='coerce'
)

df = df.drop_duplicates(subset=['Region/Country/Area', 'Year', 'Series'])

# 2. Seleccionar el ano 2020 y evitar mezclar paises con totales regionales.
# Son codigos de regiones, totales y grupos especiales de este mismo archivo.
codigos_agrupados = [
    1, 2, 15, 202, 14, 17, 18, 11, 19, 21, 419, 29, 13, 5,
    142, 143, 30, 62, 35, 34, 145, 150, 151, 154, 39, 155,
    9, 53, 54, 57, 61, 432, 199, 722
]

df_2020 = df[(df['Year'] == 2020) &
             (~df['Region/Country/Area'].isin(codigos_agrupados))].copy()

# 3. Colocar cada indicador en una columna (una fila por pais).
datos = df_2020.pivot_table(
    index='Pais', columns='Series', values='Value', aggfunc='first'
)

datos = datos.rename(columns={
    'Infant mortality for both sexes (per 1,000 live births)': 'Mortalidad infantil',
    'Life expectancy at birth for both sexes (years)': 'Esperanza de vida',
    'Life expectancy at birth for males (years)': 'Esperanza hombres',
    'Life expectancy at birth for females (years)': 'Esperanza mujeres',
    'Maternal mortality ratio (deaths per 100,000 population)': 'Mortalidad materna',
    'Population annual rate of increase (percent)': 'Crecimiento poblacional',
    'Total fertility rate (children per women)': 'Fecundidad'
})

# Quitar filas incompletas solamente para la grafica de dispersion.
dispersion = datos[['Mortalidad infantil', 'Esperanza de vida']].dropna()

print('REGISTROS DEL ANALISIS DE 2020')
print('Paises con datos en ambas variables:', len(dispersion))
print('\nResumen de ambas variables:')
print(dispersion.describe().round(2))

# 4. Correlacion de Pearson entre mortalidad infantil y esperanza de vida.
r = dispersion['Mortalidad infantil'].corr(dispersion['Esperanza de vida'])
print('\nCorrelacion entre mortalidad infantil y esperanza de vida:', round(r, 3))

sns.set_theme(style='whitegrid')
salida = carpeta / 'graficos_2020'
salida.mkdir(exist_ok=True)

# GRAFICA 1 - Dispersion con linea de regresion
fig, ax = plt.subplots(figsize=(10, 6.4))
sns.regplot(
    data=dispersion, x='Mortalidad infantil', y='Esperanza de vida',
    scatter_kws={'alpha': 0.70, 's': 52, 'color': '#2676A7'},
    line_kws={'color': '#E46B58', 'linewidth': 2.5},
    ax=ax
)
ax.set_title('Mortalidad infantil y esperanza de vida | 2020', fontsize=16, weight='bold', pad=16)
ax.set_xlabel('Mortalidad infantil (por cada 1,000 nacidos vivos)')
ax.set_ylabel('Esperanza de vida al nacer (anos)')
ax.text(0.98, 0.96, f'r de Pearson = {r:.3f}\nPaises = {len(dispersion)}',
        transform=ax.transAxes, ha='right', va='top',
        bbox={'boxstyle': 'round,pad=0.55', 'facecolor': 'white', 'edgecolor': '#CCD9E2'})
fig.tight_layout()
fig.savefig(salida / '01_dispersion_mortalidad_esperanza.png', dpi=200, bbox_inches='tight')
plt.show()

# 5. Correlaciones entre los siete indicadores numericos de 2020.
# Pandas utiliza los datos disponibles de cada pareja de variables.
correlaciones = datos.corr(method='pearson')
print('\nMATRIZ DE CORRELACION (2020)')
print(correlaciones.round(3))

# GRAFICA 2 - Mapa de calor con los coeficientes
fig, ax = plt.subplots(figsize=(11, 8.3))
sns.heatmap(
    correlaciones, annot=True, fmt='.2f', cmap='RdBu_r',
    vmin=-1, vmax=1, center=0, square=True,
    linewidths=0.5, linecolor='white',
    annot_kws={'size': 9}, ax=ax,
    cbar_kws={'label': 'Correlacion de Pearson'}
)
ax.set_title('Correlaciones entre indicadores demograficos | 2020',
             fontsize=15, weight='bold', pad=17)
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right')
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
fig.tight_layout()
fig.savefig(salida / '02_mapa_calor_correlaciones.png', dpi=200, bbox_inches='tight')
plt.show()

# 6. Identificar las parejas mas correlacionadas (sin repetir ni incluir 1.0 consigo misma).
pares = []
columnas = correlaciones.columns
for i in range(len(columnas)):
    for j in range(i + 1, len(columnas)):
        valor = correlaciones.iloc[i, j]
        pares.append([columnas[i], columnas[j], valor, abs(valor)])

ranking = pd.DataFrame(
    pares,
    columns=['Variable 1', 'Variable 2', 'Correlacion', 'Valor absoluto']
)
ranking = ranking.sort_values('Valor absoluto', ascending=False).reset_index(drop=True)
print('\nLAS 10 PAREJAS MAS CORRELACIONADAS')
print(ranking[['Variable 1', 'Variable 2', 'Correlacion']].head(10).round(3).to_string(index=False))

# GRAFICA 3 - Ranking de las ocho relaciones mas fuertes
principales = ranking.head(8).copy()
principales['Relacion'] = principales['Variable 1'] + ' / ' + principales['Variable 2']
principales = principales.iloc[::-1]
colores = ['#C65D58' if x < 0 else '#2D7E9B' for x in principales['Correlacion']]
fig, ax = plt.subplots(figsize=(12, 6.5))
barras = ax.barh(principales['Relacion'], principales['Correlacion'], color=colores)
ax.axvline(0, color='#394855', linewidth=1)
ax.set_xlim(-1.12, 1.12)
ax.set_xlabel('Coeficiente de correlacion de Pearson (r)')
ax.set_title('Relaciones lineales mas fuertes entre indicadores | 2020',
             fontsize=15, weight='bold', pad=15)
for barra, valor in zip(barras, principales['Correlacion']):
    x = valor - 0.035 if valor < 0 else valor + 0.035
    ax.text(x, barra.get_y() + barra.get_height() / 2, f'{valor:.3f}',
            ha='right' if valor < 0 else 'left', va='center', fontsize=9)
fig.tight_layout()
fig.savefig(salida / '03_ranking_correlaciones.png', dpi=200, bbox_inches='tight')
plt.show()

# Guardar resultados en CSV por si se necesitan para el reporte.
correlaciones.round(4).to_csv(salida / 'matriz_correlaciones_2020.csv', encoding='utf-8-sig')
ranking.round(4).to_csv(salida / 'ranking_correlaciones_2020.csv', index=False, encoding='utf-8-sig')
print('\nGraficas y tablas guardadas en:', salida)
